# 01. A small set

**Tier:** Deep. **Why you'd use this:** You need a bag of integers that supports add, remove, and "pick a random member" so each member is equally likely, all in expected constant time.

**After this lesson you can:** back a set with a list plus a dict of indexes; swap-remove so deletion stays fast.
About 25 minutes. **Uses:** [Dicts](../../course/04-collections/03-dicts-lesson.ipynb), [Classes](../../course/11-classes/01-classes-lesson.ipynb)


## Quick review

1. Why is removing from the middle of a list alone too slow for this API?
2. What does the dict store next to the list?

<details><summary>Answers</summary>

Shifting the tail is linear. The dict maps each value to its index in the list so you can find and swap in constant time.
</details>


## Recognize it

The prompt defines a small class API: insert, delete, membership, and sometimes get-random, each with a tight time bound.

Signals: "design a data structure", listed methods, and average O(1) requirements. A look-alike that is not this pattern: a pure algorithm on a fixed array with no API to maintain under updates.

If get-random is absent, a bare `set` usually wins. The list-plus-dict pairing exists to serve uniform random draws beside the other three methods.


## Why it works this way

A Python `set` gives average-constant add, remove, and contains, but it does not offer uniform random access to an element. A list gives random index access, but finding or removing an arbitrary value is linear. Pair them: keep the members in a list, and keep a dict from value to its index in that list.

Add appends to the list and records the index. Contains is a dict lookup. Remove finds the index, swaps the target with the last list element, updates the moved element's index, then pops the tail. That swap-remove avoids shifting the middle of the list. Get-random draws `random.randrange(len(list))` and returns that slot — each member has the same chance because the list holds each member once.

The design reason is the mismatch between the operations. Hash tables answer "where is this key?" Lists answer "what is at this index?" Neither structure alone meets all four methods in expected constant time. See [Dicts](../../internals/04-dicts/01-hash-table-lesson.ipynb) for average-constant key lookup, and [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why pop from the end is cheap while delete from the middle is not.

An `OrderedDict` LRU cache is a different design problem: recency order and eviction, not uniform random draws. A plain `set` is enough when you never need get-random. The list-plus-dict pair is the usual answer when the challenge names all four methods together.

Marker for uniqueness: 15-design:01-tiny-set-lesson.


## When to use it

Use list-plus-index-dict when the API needs fast add, remove, contains, and uniform get-random. Prefer a bare `set` when random access is not required. Prefer `OrderedDict` or a linked map when the constraint is recency eviction. Prefer a heap when you need min or max repeatedly rather than a uniform random member.

If duplicates are allowed, the dict must store multisets of indexes or you need a different design. This sibling assumes unique members, which keeps swap-remove correct.


## Brute force, and why it's slow

Remove by scanning the list and shifting the tail. Count the moves.


In [1]:
def remove_slow(values, target):
    moves = 0
    for i, value in enumerate(values):
        moves += 1
        if value == target:
            values.pop(i)
            return True, moves
    return False, moves


for size in (100, 400, 1600):
    data = list(range(size))
    print(size, remove_slow(data, size // 2))

100 (True, 51)
400 (True, 201)
1600 (True, 801)


## Key insight

Store members in a list for random picks. Store each member's index in a dict so remove can swap with the tail and pop in constant time. Update the moved member's index after the swap or the dict lies.


## Worked example

Implement a bag of unique integers with `add`, `remove`, `contains`, and `get_random`. Assume `get_random` is only called when the bag is non-empty.


In [2]:
import random


class LuckyBag:
    def __init__(self) -> None:
        # 1. values for random index access; spot for value → index.
        self.values: list[int] = []
        self.spot: dict[int, int] = {}

    def add(self, key: int) -> bool:
        if key in self.spot:
            return False
        self.spot[key] = len(self.values)
        self.values.append(key)
        return True

    def remove(self, key: int) -> bool:
        if key not in self.spot:
            return False
        # 2. Swap with the tail, then pop.
        index = self.spot[key]
        last = self.values[-1]
        self.values[index] = last
        self.spot[last] = index
        self.values.pop()
        del self.spot[key]
        return True

    def contains(self, key: int) -> bool:
        return key in self.spot

    def get_random(self) -> int:
        return self.values[random.randrange(len(self.values))]


bag = LuckyBag()
print(bag.add(1), bag.add(2), bag.add(1))
print(bag.remove(1), bag.contains(1), bag.contains(2))
print(bag.get_random())

True True False
True False True
2


## Trace

Start empty. `add(1)`, `add(2)`, `remove(1)`:

| op | values | spot |
|---|---|---|
| add 1 | `[1]` | `{1: 0}` |
| add 2 | `[1, 2]` | `{1: 0, 2: 1}` |
| remove 1 | swap → `[2, 2]` then pop → `[2]` | `{2: 0}` |

`contains(1)` is `False`. `get_random()` must return `2`.


## Cost

Each method does a constant number of dict and list end operations on average. Extra memory is one list slot and one dict entry per member. Swap-remove beats shifting a list tail on every delete.


## Variants

Not solved here: [LRU Cache](../../challenges/problems/16-lru-cache.ipynb), [Time Based Key-Value](../../challenges/problems/65-time-based-key-value.ipynb), [Find Median from Data Stream](../../challenges/problems/42-find-median-from-data-stream.ipynb).


## Common mistake

This bug is silent until remove corrupts indexes. The code pops the value without swapping, leaving the dict pointing at stale indexes. After removing a middle element, `contains` may still look true while `get_random` can raise or return a ghost. Here `remove(1)` leaves `spot` wrong and the next `get_random` path is unsafe. The print shows `spot` still holding `1` after a broken remove path that only deletes from the dict when the value was last — simplified demo: deleting index without swap.


In [3]:
class LuckyBagWrong:
    def __init__(self):
        self.values = []
        self.spot = {}

    def add(self, key):
        if key in self.spot:
            return
        self.spot[key] = len(self.values)
        self.values.append(key)

    def remove(self, key):
        if key not in self.spot:
            return
        index = self.spot.pop(key)
        self.values.pop(index)  # shifts tail; spot indexes go stale
        # spot for moved values never updated


bag = LuckyBagWrong()
bag.add(1)
bag.add(2)
bag.add(3)
bag.remove(1)
print(bag.values, bag.spot)

[2, 3] {2: 1, 3: 2}


The fix swaps with the last element and updates that element's index before popping. The worked example does that, so after removing `1` from `{1,2,3}` the structure stays consistent.


In [4]:
bag = LuckyBag()
bag.add(1)
bag.add(2)
bag.add(3)
bag.remove(1)
print(bag.values, bag.spot, bag.contains(1), bag.contains(3))

[3, 2] {2: 1, 3: 0} False True


## Python idioms

`random.choice(self.values)` also works for get-random. `dict.pop` can remove a key in one call when you already trust membership. A simpler practice set can wrap a Python `set` when get-random is out of scope.


In [5]:
class TinySet:
    def __init__(self) -> None:
        self.data: set[int] = set()

    def add(self, key: int) -> None:
        self.data.add(key)

    def remove(self, key: int) -> None:
        self.data.discard(key)

    def contains(self, key: int) -> bool:
        return key in self.data


box = TinySet()
box.add(1)
box.remove(1)
print(box.contains(1))

False


## Recap

- List holds members for random index picks.
- Dict holds indexes so remove can swap-pop.
- Stale indexes after a middle pop are the usual bug.

Next: [practice](01-tiny-set-practice.ipynb).
